# 6章 サンプル（ライブラリあり版） ― numpyで5次元→2次元（PCA）

06-1Reduce.html の中身と、まったく同じデータ・同じ手順（標準化 → 共分散行列 → 固有値問題 → 上位2本 → 射影）で、
5つの諸元を2つの軸（PC1・PC2）に落とし込む。固有値・固有ベクトルの計算だけ、htmlの自作Jacobi法ではなく、
`numpy.linalg.eigh` を使う。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Colabで日本語が文字化けしないようにするためのおまじない
try:
    import japanize_matplotlib
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'japanize-matplotlib'])
    import japanize_matplotlib

# 06-1Reduce.html の defaultData と完全に同じデータ
car_names = ['軽自動車', 'コンパクト', 'セダン', 'SUV', 'スポーツカー']
car_colors = ['#C1502E', '#4A6D7C', '#8A7A3F', '#6B4E8E', '#3E7A5B']
spec_names = ['排気量(L)', '馬力(PS)', '重量(kg)', '燃費(km/L)', '価格(万円)']

X = np.array([
    [0.66,  52,  780, 21, 150],   # 軽自動車
    [1.5,  110, 1050, 18, 220],   # コンパクト
    [2.0,  160, 1400, 15, 320],   # セダン
    [2.5,  220, 1800, 11, 420],   # SUV
    [3.5,  350, 1500,  8, 800],   # スポーツカー
], dtype=float)

X


## 1. 標準化

各諸元は単位もスケールもバラバラ（排気量は0〜4程度、価格は100〜800）なので、平均0・分散1にそろえる。

$$z_{ij} = \frac{x_{ij} - \bar{x}_j}{\sigma_j}$$

（htmlのcolStats/transformMatrixに合わせ、標準偏差は母標準偏差＝サンプル数で割ったものを使う）


In [ ]:
mean = X.mean(axis=0)
std = X.std(axis=0, ddof=0)   # 母標準偏差。html版のcolStatsと同じ(nで割る)

Z = (X - mean) / std

print("平均:", np.round(mean, 3))
print("標準偏差:", np.round(std, 3))
print("標準化後:\n", np.round(Z, 3))


## 2. 共分散行列

$$C = \frac{1}{n-1} Z^\top Z$$

（htmlのcovariance()と同じく、$(n-1)$で割った正式な共分散行列にする）


In [ ]:
n = X.shape[0]
C = (Z.T @ Z) / (n - 1)

print("共分散行列 C:\n", np.round(C, 4))


## 3. 固有値問題を解く（ライブラリを使う部分）

$$C w = \lambda w$$

`C`は対称行列なので、対称行列専用で数値的に安定した`np.linalg.eigh`を使う。


In [ ]:
values, vectors = np.linalg.eigh(C)

# 固有値の大きい順に並べ替える
order = np.argsort(values)[::-1]
values = values[order]
vectors = vectors[:, order]

# 符号をそろえる（各固有ベクトルの絶対値最大の成分をプラスにする。html版と同じ規則）
for k in range(vectors.shape[1]):
    idx = np.argmax(np.abs(vectors[:, k]))
    if vectors[idx, k] < 0:
        vectors[:, k] *= -1

ratio = np.maximum(values, 0) / np.maximum(values, 0).sum()

print("固有値(降順):", np.round(values, 4))
print("寄与率(%):", np.round(ratio * 100, 2))


## 4. 上位2本を採用して、5次元→2次元に射影する

$$\text{score}_{i} = Z_i \cdot w_1,\ Z_i \cdot w_2$$


In [ ]:
W = vectors[:, :2]          # 5次元→2次元にする係数（PC1・PC2の固有ベクトル）
scores = Z @ W               # 各車のPC1・PC2上の座標

# バイプロットの矢印用（各諸元がPC1・PC2にどれだけ効いているか）
loadings = W * np.sqrt(np.maximum(values[:2], 0))

print("W (5x2):\n", np.round(W, 3))
print("\nscores (5x2):\n", np.round(scores, 3))


## 5. 可視化（06-1Reduce.htmlのバイプロットと同じ見せ方）

車には名前（軽自動車、コンパクトなど）を、矢印には諸元名をラベルとして付ける。


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

# 諸元の矢印（バイプロットのloading）
arrow_scale = 3.0
for j, name in enumerate(spec_names):
    ax.annotate('', xy=(loadings[j, 0]*arrow_scale, loadings[j, 1]*arrow_scale), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='#7d746a', lw=1.2))
    ax.text(loadings[j, 0]*arrow_scale*1.1, loadings[j, 1]*arrow_scale*1.1,
            name.split('(')[0], color='#7d746a', fontsize=9)

# 車の点＋名前
for i, name in enumerate(car_names):
    ax.scatter(scores[i, 0], scores[i, 1], color=car_colors[i], s=70, zorder=3)
    ax.annotate(name, (scores[i, 0], scores[i, 1]), xytext=(6, 6),
                textcoords='offset points', fontsize=10)

ax.axhline(0, color='#D8D3C7', lw=1)
ax.axvline(0, color='#D8D3C7', lw=1)
ax.set_xlabel(f'PC1  ({ratio[0]*100:.1f}%)')
ax.set_ylabel(f'PC2  ({ratio[1]*100:.1f}%)')
ax.set_title('PCA: PC1 × PC2（バイプロット） ― ライブラリ版(eigh)')
ax.set_aspect('equal')
plt.tight_layout()
plt.show()
